In [2]:
!pip install rasterio s3fs matplotlib numpy geopandas folium
!pip install sentinelsat 
!pip install sentinelhub  
!pip install planet-client 


   ---------------------------------------- 0/4 [html2text]
   -------------------- ------------------- 2/4 [geomet]
   ------------------------------ --------- 3/4 [sentinelsat]
   ---------------------------------------- 4/4 [sentinelsat]


   ---------------------------------------- 0/5 [aenum]
   ---------------------------------------- 0/5 [aenum]
   -------- ------------------------------- 1/5 [utm]
   ------------------------ --------------- 3/5 [tifffile]
   -------------------------------- ------- 4/5 [sentinelhub]
   -------------------------------- ------- 4/5 [sentinelhub]
   -------------------------------- ------- 4/5 [sentinelhub]
   -------------------------------- ------- 4/5 [sentinelhub]
   -------------------------------- ------- 4/5 [sentinelhub]
   -------------------------------- ------- 4/5 [sentinelhub]
   -------------------------------- ------- 4/5 [sentinelhub]
   ---------------------------------------- 5/5 [sentinelhub]



ERROR: Could not find a version that satisfies the requirement planet-client (from versions: none)
ERROR: No matching distribution found for planet-client


In [3]:
# run in Jupyter
import rasterio
import numpy as np
import matplotlib.pyplot as plt
import s3fs

# Example: a public Sentinel-2 L2A COG URL on AWS (you will replace this with the S3 path you need)
# You can browse the AWS Registry of Open Data for exact keys. See citation.
cog_s3_url = "s3://sentinel-s2-l2a-cogs/tiles/33/U/UP/2025/8/26/0/B04.tif"  # example — replace with actual tile/date

fs = s3fs.S3FileSystem(anon=True)  # anonymous access to public bucket

def read_band(path):
    with fs.open(path, "rb") as f:
        with rasterio.MemoryFile(f.read()) as mem:
            with mem.open() as src:
                return src.read(1).astype('float32'), src.transform

# Paths for B04 (red), B03 (green), B02 (blue) for the same tile/date
b4_path = "s3://sentinel-s2-l2a-cogs/tiles/33/U/UP/2025/8/26/0/B04.tif"
b3_path = "s3://sentinel-s2-l2a-cogs/tiles/33/U/UP/2025/8/26/0/B03.tif"
b2_path = "s3://sentinel-s2-l2a-cogs/tiles/33/U/UP/2025/8/26/0/B02.tif"

b4, t = read_band(b4_path)
b3, _ = read_band(b3_path)
b2, _ = read_band(b2_path)

# simple stretch to 0-1 for display (uint16 -> scale)
def scale_band(b):
    b = np.nan_to_num(b)
    p2, p98 = np.percentile(b, (2, 98))
    return np.clip((b - p2) / (p98 - p2), 0, 1)

rgb = np.dstack([scale_band(b4), scale_band(b3), scale_band(b2)])

plt.figure(figsize=(10,10))
plt.imshow(rgb)
plt.axis('off')
plt.title('Sentinel-2 true color (B4,B3,B2)')
plt.show()


FileNotFoundError: The specified bucket does not exist